# BERT와 ELECTRA 모델 비교 실습

- 이번 복습과제에서는 SST-2 데이터셋을 기반으로 BERT와 ELECTRA 모델을 학습시켜보고 성능과 구조의 차이를 알아보겠습니다.
- 코드 실행시간이 매우 길 수 있습니다.
  - 최대한 끝까지 실행해보시되, 시간 부족으로 인해 중간에 중지하신 실행 결과를 제출하셔도 괜찮습니다.
  - 제출 이후에는 꼭 끝까지 실행시켜 비교해보시기 바랍니다!

In [1]:
!pip install --upgrade --quiet datasets fsspec huggingface_hub

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 491.5/491.5 kB 11.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 193.6/193.6 kB 11.7 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.2 requires fsspec==2025.3.2, but you have fsspec 2025.3.0 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cublas-cu12==12.4.5.8; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cublas-cu12 12.5.3.2 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-cupti-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-cupti-cu12 12.5.82 which is incompatible.
torch 2.6.0+cu124 requires nvidia-cuda-nvrtc-cu12==12.4.127; platform_system == "Linux" and platform_machine == "x86_64", but you have nvidia-cuda-nvrtc-cu12 12.5.82 which is incompatible.
torch 2.6

---------------
여기까지만 실행
---------------
그 다음,  런타임 > 세션 다시 시작 > 아래 셀부터 실행

In [2]:
import torch
from torch.utils.data import DataLoader
from transformers import AutoTokenizer, AutoModelForSequenceClassification
from datasets import load_dataset
from torch.optim import AdamW
from tqdm import tqdm

In [3]:
# batch_size와 epochs를 조정해보세요!
batch_size = 32
epochs = 3
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

Using device: cuda


In [ ]:
# 데이터셋 로드
raw_datasets = load_dataset("sst2")
raw_datasets

/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md:   0%|          | 0.00/5.27k [00:00<?, ?B/s]

train-00000-of-00001.parquet:   0%|          | 0.00/3.11M [00:00<?, ?B/s]

validation-00000-of-00001.parquet:   0%|          | 0.00/72.8k [00:00<?, ?B/s]

test-00000-of-00001.parquet:   0%|          | 0.00/148k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/67349 [00:00<?, ? examples/s]

In [5]:
# 전처리
def tokenize_function(examples, tokenizer):
    return tokenizer(examples["sentence"], padding="max_length", truncation=True, max_length=128)

## 🔹 BERT와 ELECTRA 실험

In [7]:
# 학습 함수 정의
def train_and_evaluate(model_name):
    print(f"\n======== Now Training: {model_name} ========")

    tokenizer = AutoTokenizer.from_pretrained(model_name)
    tokenized_datasets = raw_datasets.map(lambda x: tokenize_function(x, tokenizer), batched=True)

    tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
    tokenized_datasets.set_format("torch", columns=["input_ids", "attention_mask", "labels"])
    train_dataset = tokenized_datasets["train"]
    valid_dataset = tokenized_datasets["validation"]

    train_loader = DataLoader(train_dataset, shuffle=True, batch_size=32)
    valid_loader = DataLoader(valid_dataset, batch_size=32)

    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2).to(device)
    optimizer = AdamW(model.parameters(), lr=2e-5)

    model.train()
    for epoch in range(3):
        total_loss = 0
        for batch in tqdm(train_loader, desc=f"Epoch {epoch+1}"):
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(
                input_ids=batch["input_ids"],
                attention_mask=batch["attention_mask"],
                labels=batch["labels"],
            )
            loss = outputs.loss
            total_loss += loss.item()

            loss.backward()
            optimizer.step()
            optimizer.zero_grad()

        avg_loss = total_loss / len(train_loader)
        print(f"Epoch {epoch+1} - Avg Train Loss: {avg_loss:.4f}")

    model.eval()
    correct = 0
    total = 0
    with torch.no_grad():
        for batch in valid_loader:
            batch = {k: v.to(device) for k, v in batch.items()}
            outputs = model(**batch)
            predictions = torch.argmax(outputs.logits, dim=-1)
            correct += (predictions == batch["labels"]).sum().item()
            total += batch["labels"].size(0)

    acc = correct / total
    print(f"Validation Accuracy ({model_name}): {acc:.4f}")
    return acc

# 실행 및 평가
bert_acc = train_and_evaluate("bert-base-uncased")
electra_acc = train_and_evaluate("google/electra-base-discriminator")


======== Now Training: bert-base-uncased ========


Map:   0%|          | 0/872 [00:00<?, ? examples/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.
Epoch 1: 100%|██████████| 2105/2105 [21:37<00:00,  1.62it/s]


Epoch 1 - Avg Train Loss: 0.2024


...

Epoch 1 - Avg Train Loss: 0.1840


Epoch 2: 100%|██████████| 2105/2105 [21:58<00:00,  1.60it/s]


Epoch 2 - Avg Train Loss: 0.1085


Epoch 3: 100%|██████████| 2105/2105 [21:58<00:00,  1.60it/s]


Epoch 3 - Avg Train Loss: 0.0788
Validation Accuracy (google/electra-base-discriminator): 0.9495


## 📊 결과 비교 및 분석

아래 항목에 대한 답을 간략히 적어주세요:

1. 각 모델 구조 설명
2. 어떤 모델이 적합한지에 대한 본인의 의견
  - 학습 속도, accuracy 등 고려


##BERT
- Transformer 기반 모델로 2018년 Google AI에 의해 개발된 모델 -> pretrained Language Model이며, open source로 공개되었음
- 따라서 아키텍처와 코드가 잘 알려져 있고, Google은 BERT의 사전학습 모델도 인터넷에 업로드해 두었음(다운로드해서 바로 사용가능) BERT의 다양한 변형 모델들도 인터넷에 존재하며 사용법도 매우 간단

- BERT는 오직 Encoder만으로 구성되어있음 (구조 자체는 Transformer 인코더와 거의 동일 다만 디코더가 없음)
- (<-> 기존 Transformer 아키텍처는 인코더와 디코더 모두 존재)
BERT의 encoder는 bidirectional -> BERT가 학습되는 방식 자체가 양방향이기 때문

- BERT는 bidirectional context를 기반으로 표현을 학습 -> 단어 의미 이해에 매우 중용
(ex) “bank”라는 단어가 있는 두 문장 -> 첫번째 문장 ”river bank” : bank = 강둑 / 두번째 문장 “bank ~ deposit” : bank = 금융기관

- BERT 구조 자체는 특별할 것 없지만, 오직 인코더로만 구성되어있고 학습 방식이 중요함
BERT의 pretraining은 Self-supervised Learning 방식으로 수행됨
---
###MLM(masked language modeling)
- 전체 단어의 약 15% 정도를 마스킹 또는 다른 방식으로 변형함
  1.  그 중 80%는 [MASK]로 교체되고
  2.  10%는 random 단어로 바뀜 (ex) coffee같은 무관한 단어 -> 잡음을 추가하고 일반화 성능을 높이기 위한 일종의 regularization
  3.  나머지 10%는 그대로 남겨둠 -> forward propagation 시 실제 마스킹이 없는 상황을 모방하기 위한 것

- 이후 BERT가 이 문장을 주고 나서 마스킹된 단어가 무엇인지를 맞추게 함
- “이 문장을 입력으로 넣었을 때 정답라벨(Ground Truth)는 무엇인가? ” -> 정답은 “fox” 우리는 fox를 마스킹했기 때문
---
###NSP(next sentence prediction)
- BERT에 두 문장을 입력으로 넣는데 사실 이 두 문장은 하나의 입력으로 연결됨
- Supervised training을 수행하기 위해 ground truth(정답 레이블)이 필요

- 그러나, 대규모 코퍼스(교과서)에는 연속되는 문장들이 많이있음
- 이 중에서 두 문장을 연속으로 추출하여 하나는 문장A, 하나는 문장B로 구성해서 BERT에 넣으면 됨
  -> 이 경우 정답은 “next” 그래서 우리는 레이블로 ”next”를 부여
---
## ELECTRA(Efficiently Learning an Encoder that Classifies Token Replacements Accurately)
- pretraining : Replaced Token Detection
- Generator가 바궈 놓은 토큰을 Discriminator가 진짜/가짜 판별
- 모든 토큰에 대해 학습 신호 제공 -> 더 빠르고 효과적

---
## 두 모델의 학습 성능 비교
- Epoch당 소요 시간
  - BERT :약 21분 40초
  - ELECTRA : 약 21분 59초

- Train Loss(3 Epochs)
  - BERT : 0.2024 > 0.0714
  - ELECTRA : 0.1840 > 0.0788

- Validation Accuracy
  - BERT : 0.9220
  - ELECTRA : 0.9495

- 수렴 속도
  - BERT : loss가 감소하지만 느리게 감소
  - ELECTRA : loss 빠르게 감소, 정확도 빠르게 상승

- **더 적합한 모델은? : ELECTRA**
  - Replaced Token Detection 방식으로 인해 전체 토큰 학습으로 인해 더 많은 학습 신호 제공
  - 같은 학습 시간 대비 더 높은 정확도와 빠른 수렴 (ELECTRA는 94.95%의 정확도)